# Load required Libraries

In [1]:
# Data handling
import pandas as pd  # data loading, cleaning, analysis

# Web app framework
import streamlit as st  # interactive dashboard UI

# Visualization libraries
import plotly.express as px  # high-level interactive charts
import plotly.graph_objects as go  # advanced/custom visualizations

# Date & time utilities
import datetime  # working with dates and timestamps

# Image processing
from PIL import Image  # image loading and manipulation (via Pillow)
import warnings  # built-in module for handling warning messages
warnings.filterwarnings("ignore")  # suppress all warnings for cleaner output (useful in dashboards)


# Streamlit Dashboard for Customer Sentiment Analysis

In [2]:
@st.cache_data
def load_data(path: str) -> pd.DataFrame:
    """
    DATA PIPELINE FOR SENTIMENT DASHBOARD
    --------------------------------------
    Purpose:
    Loads raw CSV and transforms it into a clean,
    analysis-ready dataset for Streamlit dashboards.

    This pipeline ensures:
    - Data consistency
    - No missing critical fields
    - Safe time-series analysis
    - Standardized sentiment labels
    """

    # ==============================================================
    # 1. LOAD RAW DATA
    # ==============================================================
    df = pd.read_csv(path)

    # ==============================================================
    # 2. VALIDATE REQUIRED COLUMNS
    # Purpose:
    # Prevents runtime crashes if dataset schema changes.
    # ==============================================================
    required_cols = ["sentiment", "country", "timestamp", "product_category"]

    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        st.error(f"Missing required columns: {missing_cols}")
        st.stop()

    # ==============================================================
    # 3. REMOVE DUPLICATES
    # Purpose:
    # Ensures each record is unique to avoid bias in analysis.
    # ==============================================================
    df = df.drop_duplicates()

    # ==============================================================
    # 4. STANDARDISE SENTIMENT LABELS
    # Purpose:
    # Converts inconsistent labels (e.g. POSITIVE, positive)
    # into a clean format:
    # → Positive / Negative / Neutral
    # ==============================================================
    df["sentiment"] = (
        df["sentiment"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "positive": "Positive",
            "negative": "Negative",
            "neutral": "Neutral"
        })
    )

    # Remove rows where sentiment could not be mapped
    df = df.dropna(subset=["sentiment"])

    # ==============================================================
    # 5. CLEAN TEXT FIELDS (COUNTRY + CATEGORY)
    # Purpose:
    # Removes whitespace and invalid placeholder values.
    # ==============================================================
    df["country"] = df["country"].astype(str).str.strip()
    df["product_category"] = df["product_category"].astype(str).str.strip()

    df = df[
        (df["country"] != "")
        & (df["product_category"] != "")
        & (df["country"].str.lower() != "nan")
    ]

    # ==============================================================
    # 6. PARSE TIMESTAMP COLUMN
    # Purpose:
    # Converts string dates into datetime format for trend analysis.
    # Invalid dates become NaT (Not a Time).
    # ==============================================================
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)

    # Remove rows where timestamp is invalid
    df = df.dropna(subset=["timestamp"])

    # ==============================================================
    # 7. FEATURE ENGINEERING (TIME-BASED FEATURES)
    # Purpose:
    # Enables time-series analysis and reporting.
    # ==============================================================
    df["date"] = df["timestamp"].dt.date        # daily trends
    df["year"] = df["timestamp"].dt.year        # yearly aggregation
    df["month"] = df["timestamp"].dt.to_period("M").astype(str)  # monthly trends

    # ==============================================================
    # 8. HANDLE MISSING TEXT DATA
    # Purpose:
    # Ensures "processed_review" always exists for display or NLP.
    # ==============================================================
    if "processed_review" not in df.columns:
        df["processed_review"] = ""

    if "_clean_text" in df.columns:
        df["processed_review"] = df["processed_review"].fillna(df["_clean_text"])

    # ==============================================================
    # 9. FINAL VALIDATION CHECK
    # Purpose:
    # Ensures pipeline does not return an empty dataset.
    # ==============================================================
    if len(df) == 0:
        st.error("Dataset is empty after cleaning. Check raw input file.")
        st.stop()

    return df

# ==============================================================
# LOAD DATA
# Purpose:
# Load the cleaned dataset from CSV into a pandas DataFrame.
# ==============================================================

df = load_data("processed_sentiment_data.csv")


# ==============================================================
# APP TITLE
# Purpose:
# Display main dashboard title in Streamlit UI.
# ==============================================================

st.title("📊 Customer Sentiment Dashboard")


# ==============================================================
# SIDEBAR FILTERS
# Purpose:
# Allow users to dynamically filter data by country and category.
# ==============================================================

st.sidebar.header("Filters")

# Extract unique filter options from dataset
countries = sorted(df["country"].unique())
categories = sorted(df["product_category"].unique())

# Multi-select dropdown for country filtering
selected_country = st.sidebar.multiselect(
    "Country",
    countries,
    default=countries
)

# Multi-select dropdown for product category filtering
selected_category = st.sidebar.multiselect(
    "Product Category",
    categories,
    default=categories
)

# Apply filters to dataset
filtered_df = df[
    df["country"].isin(selected_country) &
    df["product_category"].isin(selected_category)
]


# ==============================================================
# KPI METRICS
# Purpose:
# Display high-level business performance indicators.
# ==============================================================

st.header("📊 Key Metrics")

total = len(filtered_df)

# Calculate sentiment percentages
positive = (filtered_df["sentiment"] == "Positive").mean() * 100
negative = (filtered_df["sentiment"] == "Negative").mean() * 100
neutral = (filtered_df["sentiment"] == "Neutral").mean() * 100

# Create KPI layout (4 columns)
col1, col2, col3, col4 = st.columns(4)

# Display metrics in dashboard cards
col1.metric("Total Reviews", total)
col2.metric("Positive %", f"{positive:.1f}%")
col3.metric("Neutral %", f"{neutral:.1f}%")
col4.metric("Negative %", f"{negative:.1f}%")


# ==============================================================
# PIE CHART (SENTIMENT DISTRIBUTION)
# Purpose:
# Show proportion of each sentiment category visually.
# ==============================================================

st.header("🍩 Sentiment Distribution")

# Count occurrences of each sentiment
sentiment_dist = filtered_df["sentiment"].value_counts().reset_index()
sentiment_dist.columns = ["Sentiment", "Count"]

# Create donut chart visualization
fig = px.pie(
    sentiment_dist,
    names="Sentiment",
    values="Count",
    hole=0.5
)

st.plotly_chart(fig, use_container_width=True)


# ==============================================================
# BAR CHART (COUNTRY ANALYSIS)
# Purpose:
# Compare sentiment distribution across countries.
# ==============================================================

st.header("🌍 Sentiment by Country")

# Group data by country and sentiment
country_chart = filtered_df.groupby(
    ["country", "sentiment"]
).size().reset_index(name="count")

# Create grouped bar chart
fig = px.bar(
    country_chart,
    x="country",
    y="count",
    color="sentiment",
    barmode="group"
)

st.plotly_chart(fig, use_container_width=True)


# ==============================================================
# DATA TABLE
# Purpose:
# Allow users to inspect raw filtered dataset.
# ==============================================================

st.header("📋 Raw Data")

st.dataframe(filtered_df, use_container_width=True)

2026-07-01 09:50:16.527 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-01 09:50:16.537 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-01 09:50:16.545 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-01 09:50:18.038 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-01 09:50:21.207 
  command:

    streamlit run C:\Users\EUGENE\anaconda3\envs\anaconda-nlp\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-07-01 09:50:21.208 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-01 09:50:21.210 Thread 'MainThread': missing ScriptRunContext! This warning can be ig

DeltaGenerator()

# Generate business insights from sentiment analysis results